# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [mlcroissant](https://mlcommons.org/croissant/) library. The dataset contains records for human proteins analyzed by mass spectrometry.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset loaded: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In Croissant datasets, data is organized in record sets, each with its own `@id`. We'll enumerate these to explore the available data structures.

In [ ]:
# List all record sets and their available fields/columns

print("Record sets found in this dataset:\n")
for rs in dataset.record_sets():
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    # List fields/columns inside each record set
    print(f"  Fields/Columns in record set '{rs.name}':")
    for field in rs.fields:
        print(f"    - Field name: {field.name}, @id: {field.id}, Type: {field.data_type}")
    print("\n")

## 3. Data Extraction
Let's load data from a key record set into a DataFrame for analysis. We'll use the primary record set and field `@id`s from the overview above.

> ⚠️ **Note:** Choose the main record set containing protein measurements. In this dataset it's typically named `cr:ProteinRecordSet` (ID: `cr:ProteinRecordSet`). Adjust as needed if overview lists a different or extended `@id`.

In [ ]:
# Select the main record set by its @id
main_record_set_id = 'cr:ProteinRecordSet'  # replace with actual @id if different

# List of all record set @ids in the dataset (adjusted from previous cell if needed)
record_sets_ids = [rs.id for rs in dataset.record_sets()]

# Load each record set into a DataFrame
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if main_record_set_id in dataframes:
    print(f"Columns in {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print(f"Main record set {main_record_set_id} not found. Available IDs: {list(dataframes.keys())}")

## 4. Exploratory Data Analysis (EDA)
We apply common data processing steps, such as filtering records by abundance, normalizing numeric fields, and grouping by biological sample or protein type. Please use the field `@id`s from the overview section.

In [ ]:
# Choose a numeric field (by column @id) for EDA, e.g. 'cr:Abundance' (protein abundance)
numeric_field_id = 'cr:Abundance'  # replace with the correct field @id if needed, e.g. from ['cr:Abundance', 'cr:MolecularWeight', ...]

# Filter for proteins with abundance greater than a threshold (e.g. 10)
threshold = 10
df = dataframes[main_record_set_id]
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize the selected numeric field
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} (z-score):")
    display(filtered_df[[numeric_field_id, col_norm]].head())

    # Group by another field, e.g. protein type or sample (replace with actual @id)
    group_field_id = 'cr:Sample'  # Replace with a field @id present in dataset, e.g. 'cr:Sample', 'cr:ProteinType', etc.
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
    else:
        print(f"Group field {group_field_id} not present in columns. Columns: {df.columns.tolist()}")
else:
    print(f"Numeric field {numeric_field_id} not found in columns.\nColumns: {df.columns.tolist()}")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset. Here we plot the normalized abundance distribution after filtering, and abundance by sample/group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of normalized abundance
if numeric_field_id in filtered_df:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[col_norm], bins=30, kde=True)
    plt.title(f"Distribution of normalized {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id} (z-score)")
    plt.show()

# Boxplot by group
if group_field_id in filtered_df:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We have loaded and explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. After reviewing available record sets and fields using their `@id`s, we extracted records, filtered proteins by abundance, normalized and grouped data, and visualized key relationships.

This workflow provides a foundation for downstream analysis, including protein biomarker discovery, protein comparison across samples, or further biological interpretation. For advanced tasks, refer to the [mlcroissant documentation](https://github.com/mlcommons/croissant) for more advanced dataset manipulation and querying features.